# Regroup Positives clusters for threshold

In [22]:
import pandas as pd
import glob
import os

# Définition du dossier contenant les fichiers
folder_path = "U:/PhD/Experimentation/Electrophysiology/S1_recordings_Analysis/spyking_analysis/reponse_data"

# Recherche des fichiers correspondant au motif "Neuron_resp_*_allego_*.xlsx"
file_list = glob.glob(os.path.join(folder_path, "Neuron_resp_*_allego_*.xlsx"))

# Initialisation d'une liste pour stocker les DataFrames filtrés
filtered_data = []

# Parcours de tous les fichiers trouvés
for file in file_list:
    df = pd.read_excel(file)  # Lecture du fichier
    
    # Vérification si la colonne 'response' existe
    if "response" in df.columns:
        df_filtered = df[df["response"] == 1]  # Filtrage des lignes où response == 1
        filtered_data.append(df_filtered)  # Ajout au DataFrame final

# Vérification si on a des données à sauvegarder
if filtered_data:
    # Concatenation de tous les DataFrames filtrés
    final_df = pd.concat(filtered_data, ignore_index=True)

    # Définition du fichier de sortie
    list_output_file = os.path.join(folder_path, "list_valid_threhold_clusters.xlsx")

    # Sauvegarde en fichier Excel
    final_df.to_excel(list_output_file, index=False)
    print(f"Fichier enregistré : {list_output_file}")
else:
    print("Aucune donnée à sauvegarder (aucune ligne avec response = 1 trouvée).")


Fichier enregistré : U:/PhD/Experimentation/Electrophysiology/S1_recordings_Analysis/spyking_analysis/reponse_data\list_valid_threhold_clusters.xlsx


Spression de la collonne neuron_id.1 en trop

# Ajout des infos clusters

In [23]:
import pandas as pd
import os
import re

# Définition des paths principaux
converter_path = "U:/PhD/Experimentation/Electrophysiology/S1_recordings_Analysis/Converter_circus"
list_output_file = list_output_file

# Charger le fichier des clusters valides
df_clusters = pd.read_excel(list_output_file)

# Initialisation des nouvelles colonnes si elles n'existent pas encore
columns_to_add = ["amp", "ch", "depth", "fr", "group", "n_spikes", "purity", "sh"]
for col in columns_to_add:
    if col not in df_clusters.columns:
        df_clusters[col] = None

# Fonction pour extraire le numéro de cluster à partir de neuron_id (ex: "temp_13" -> 13)
def extract_cluster_number(neuron_id):
    match = re.search(r'temp_(\d+)', str(neuron_id))
    return match.group(1) if match else None

# Parcours de chaque ligne du fichier clusters
for index, row in df_clusters.iterrows():
    directory = row["directory"]
    file_name = row["file_name"]
    
    # Extraire le numéro de cluster spécifique pour cette ligne
    neuron_id = row["neuron_id"]
    cluster_number = extract_cluster_number(neuron_id)

    if cluster_number:
        # Construction du chemin vers le fichier "cluster_info.tsv"
        cluster_info_path = os.path.join(converter_path, directory, file_name, file_name + "_spy", file_name + "_spy.GUI", "cluster_info.tsv")
        
        # Vérification de l'existence du fichier cluster_info.tsv
        if os.path.exists(cluster_info_path):
            # Charger le fichier "cluster_info.tsv"
            df_cluster_info = pd.read_csv(cluster_info_path, sep='\t')
            
            # Filtrer pour récupérer uniquement la ligne correspondant au cluster_number actuel
            df_matching_cluster = df_cluster_info[df_cluster_info["cluster_id"].astype(str) == str(cluster_number)]

            # Si on trouve une correspondance, mettre à jour les valeurs dans df_clusters
            if not df_matching_cluster.empty:
                for col in columns_to_add:
                    df_clusters.at[index, col] = df_matching_cluster.iloc[0][col]
            else:
                print(f"Aucune correspondance trouvée pour {file_name} avec le cluster_id: {cluster_number}")
        else:
            print(f"Cluster info file not found for {file_name}")
    else:
        print(f"Numéro de cluster non trouvé pour neuron_id: {neuron_id}")

# ✅ Réorganiser les colonnes : "depth" doit être après "ch"
column_order = list(df_clusters.columns)  # Récupérer l'ordre actuel des colonnes
if "ch" in column_order and "fr" in column_order and "depth" in column_order:
    column_order.remove("depth")  # Supprimer "depth" de sa position actuelle
    index_ch = column_order.index("ch")
    column_order.insert(index_ch + 1, "depth")  # Insérer "depth" après "ch"

df_clusters = df_clusters[column_order]  # Réorganiser le DataFrame avec le nouvel ordre

# ✅ Extraire la date sans modifier la colonne "directory"
df_clusters["directory_date"] = df_clusters["directory"].str.extract(r"(\d{4}\.\d{2}\.\d{2})")[0]  # Extrait uniquement YYYY.MM.DD
df_clusters["directory_suffix"] = df_clusters["directory"].str.extract(r"_(M\d+)$")[0]  # Extrait le suffixe (ex: M1, M2), sinon NaN

# ✅ Convertir en date pour le tri et gérer les NaN pour les suffixes
df_clusters["directory_date"] = pd.to_datetime(df_clusters["directory_date"], format='%Y.%m.%d', errors='coerce')
df_clusters["directory_suffix"] = df_clusters["directory_suffix"].fillna("")  # Remplacer NaN par ""

# ✅ Trier d'abord par date, puis par suffixe
df_clusters = df_clusters.sort_values(by=["directory_date", "directory_suffix"])

# ✅ Supprimer les colonnes temporaires après le tri
df_clusters = df_clusters.drop(columns=["directory_date", "directory_suffix"])

# Sauvegarde du fichier final avec toutes les colonnes conservées
list_infos_output_file = os.path.join(folder_path, "list_infos_valid_threhold_clusters.xlsx")
df_clusters.to_excel(list_infos_output_file, index=False)

print(f"Fichier mis à jour enregistré : {list_infos_output_file}")


Fichier mis à jour enregistré : U:/PhD/Experimentation/Electrophysiology/S1_recordings_Analysis/spyking_analysis/reponse_data\list_infos_valid_threhold_clusters.xlsx
